# Preliminari

In [2]:
import sys
from pathlib import Path

# if notebook is in PRIN/notebooks, parent() is PRIN
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))
print("Added to sys.path:", project_root)

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint
import time
import json

from pydantic import BaseModel
from textwrap import dedent
from IPython.display import Math

from pathlib import Path

from utils.schema_json import ReportData

from pydantic import BaseModel
from time import perf_counter
from openai.types.fine_tuning import SupervisedMethod, SupervisedHyperparameters

Added to sys.path: C:\Users\lucat\VSCodeProjects\PRIN


In [3]:
load_dotenv()  # Load environment variables from .env file

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri

In [4]:
TRAIN_FILE_NAME = "data_finetune_hints_openai_train.jsonl"
VALIDATION_FILE_NAME = "data_finetune_hints_openai_val.jsonl"
#MODEL = "gpt-4.1-2025-04-14"
MODEL = "gpt-4.1-nano-2025-04-14" # Nano
SEED = 2025

# Upload data
Upload a file that can be used across various endpoints. Individual files can be up to 512 MB, and the size of all files uploaded by one organization can be up to 1 TB.

The Assistants API supports files up to 2 million tokens and of specific file types. See the Assistants Tools guide for details.

The Fine-tuning API only supports .jsonl files. The input also has certain required formats for fine-tuning chat or completions models.

The Batch API only supports .jsonl files up to 200 MB in size. The input also has a specific required format.

Please contact us if you need to increase these storage limits.

In [5]:
train_path = Path('../data/ft-dataset/' + TRAIN_FILE_NAME)
print(train_path)

validation_path = Path('../data/ft-dataset/' + VALIDATION_FILE_NAME)
print(validation_path)

..\data\ft-dataset\data_finetune_hints_openai_train.jsonl
..\data\ft-dataset\data_finetune_hints_openai_val.jsonl


In [6]:
if False: 
    batch_size = 5  # puoi provare anche 5 o 20
    data_batch = []
    indices = []

    with open(validation_path) as f:
        for i, line in enumerate(f):
            data = json.loads(line)
            text = " ".join([m["content"] for m in data["messages"]])
            data_batch.append(text)
            indices.append(i)

            # ogni tot righe fai una chiamata
            if len(data_batch) >= batch_size:
                done = False
                while not done:
                    try:
                        result = client.moderations.create(
                            model="omni-moderation-latest",
                            input=data_batch
                        )
                        for j, r in enumerate(result.results):
                            if r.flagged:
                                print(f"⚠️ Riga {indices[j]} flaggata:", r.categories)
                        done = True
                    except Exception as e:
                        print(f"Rate limit: attendo 10 secondi ({e})")
                        time.sleep(10)
                data_batch = []
                indices = []
    
    batch_size = 5  # puoi provare anche 5 o 20
    data_batch = []
    indices = []

    with open(train_path) as f:
        for i, line in enumerate(f):
            print(i)
            data = json.loads(line)
            text = " ".join([m["content"] for m in data["messages"]])
            data_batch.append(text)
            indices.append(i)

            # ogni tot righe fai una chiamata
            if len(data_batch) >= batch_size:
                done = False
                while not done:
                    try:
                        result = client.moderations.create(
                            model="omni-moderation-latest",
                            input=data_batch
                        )
                        for j, r in enumerate(result.results):
                            if r.flagged:
                                print(f"⚠️ Riga {indices[j]} flaggata:", r.categories)
                        done = True
                    except Exception as e:
                        print(f"Rate limit: attendo 10 secondi ({e})")
                        time.sleep(10)
                data_batch = []
                indices = []



In [7]:
# Train file
if True:
  file_object_training = client.files.create(
    file=open(train_path, "rb"),
    purpose="fine-tune",
    expires_after={
      "anchor": "created_at",
      "seconds": 2592000  # 30 days
    }
  )

In [8]:
display(file_object_training.model_dump(mode='json'))

{'id': 'file-ARyTqciMDttGF5EXrrDnW8',
 'bytes': 759187,
 'created_at': 1762106430,
 'filename': 'data_finetune_hints_openai_train.jsonl',
 'object': 'file',
 'purpose': 'fine-tune',
 'status': 'processed',
 'expires_at': 1764698430,
 'status_details': None}

In [9]:
# Validation file
if True:
  file_object_validation = client.files.create(
    file=open(validation_path, "rb"),
    purpose="fine-tune",
    expires_after={
      "anchor": "created_at",
      "seconds": 2592000  # 30 days
    }
  )

In [10]:
display(file_object_validation.model_dump(mode='json'))

{'id': 'file-YXR9TQHWqcqiZ9KXrpyNVv',
 'bytes': 39560,
 'created_at': 1762106445,
 'filename': 'data_finetune_hints_openai_val.jsonl',
 'object': 'file',
 'purpose': 'fine-tune',
 'status': 'processed',
 'expires_at': 1764698445,
 'status_details': None}

# Create a fine-tuning job
With your test data uploaded, create a fine-tuning job to customize a base model using the training data you provide. When creating a fine-tuning job, you must specify:

A base model (model) to use for fine-tuning. This can be either an OpenAI model ID or the ID of a previously fine-tuned model. See which models support fine-tuning in the model docs.
A training file (training_file) ID. This is the file you uploaded in the previous step.
A fine-tuning method (method). This specifies which fine-tuning method you want to use to customize the model. Supervised fine-tuning is the default.

In [11]:
fine_tuning_job = client.fine_tuning.jobs.create(
  training_file='file-ARyTqciMDttGF5EXrrDnW8',
  validation_file='file-YXR9TQHWqcqiZ9KXrpyNVv',
  model=MODEL,
  seed=SEED
)

In [12]:
display(client.fine_tuning.jobs.retrieve(fine_tuning_job.id).model_dump(mode='json'))

{'id': 'ftjob-XYLB0bkTj0Hay3TAiBJ3Uw4S',
 'created_at': 1762106487,
 'error': {'code': None, 'message': None, 'param': None},
 'fine_tuned_model': None,
 'finished_at': None,
 'hyperparameters': {'batch_size': 'auto',
  'learning_rate_multiplier': 'auto',
  'n_epochs': 'auto'},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'fine_tuning.job',
 'organization_id': 'org-WVBydi9dp238sOOCQoAoLWUw',
 'result_files': [],
 'seed': 2025,
 'status': 'validating_files',
 'trained_tokens': None,
 'training_file': 'file-ARyTqciMDttGF5EXrrDnW8',
 'validation_file': 'file-YXR9TQHWqcqiZ9KXrpyNVv',
 'estimated_finish': None,
 'integrations': [],
 'metadata': None,
 'method': {'type': 'supervised',
  'dpo': None,
  'reinforcement': None,
  'supervised': {'hyperparameters': {'batch_size': 'auto',
    'learning_rate_multiplier': 'auto',
    'n_epochs': 'auto'}}},
 'user_provided_suffix': None,
 'usage_metrics': None,
 'shared_with_openai': False,
 'eval_id': None}